# 🌿 Ethno-API Quickstart — Phytochemical Dataset v2.0

This notebook demonstrates how to load, explore, and analyse the
**USDA Phytochemical & Ethnobotanical Database (Enriched v2.0)**.

We use the **free 400-row sample** hosted on GitHub.  
The full dataset (104,388 records) is available at [ethno-api.com](https://ethno-api.com).

**Schema (8 columns):**

| Column | Description |
|--------|-------------|
| `chemical` | Standardised compound name |
| `plant_species` | Binomial species name |
| `application` | Traditional medicinal use |
| `dosage` | Reported dosage / concentration |
| `pubmed_mentions_2026` | PubMed publication count |
| `clinical_trials_count_2026` | ClinicalTrials.gov study count |
| `chembl_bioactivity_count` | ChEMBL bioassay data points |
| `patent_count_since_2020` | US patents since 2020 |

In [ ]:
# Cell 1 — Install dependencies (skip if already installed)
# !pip install pandas matplotlib seaborn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="viridis")
print(f"pandas {pd.__version__}")

In [ ]:
# Cell 2 — Load the free 400-row sample from GitHub
SAMPLE_URL = (
    "https://raw.githubusercontent.com/"
    "wirthal1990-tech/USDA-Phytochemical-Database-JSON/"
    "main/ethno_sample_400.json"
)

df = pd.read_json(SAMPLE_URL)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head(10)

In [ ]:
# Cell 3 — Basic statistics
print(f"Unique chemicals:  {df['chemical'].nunique()}")
print(f"Unique species:    {df['plant_species'].nunique()}")
print(f"Non-null dosages:  {df['dosage'].notna().sum()}")
print(f"Non-null applications: {df['application'].notna().sum()}")
print()
df.describe()

In [ ]:
# Cell 4 — Top 15 most-researched chemicals (by PubMed mentions)
top_pubmed = (
    df.groupby("chemical")["pubmed_mentions_2026"]
    .first()
    .nlargest(15)
)

fig, ax = plt.subplots(figsize=(10, 6))
top_pubmed.plot.barh(ax=ax, color=sns.color_palette("viridis", 15))
ax.set_xlabel("PubMed Mentions (2026)")
ax.set_title("Top 15 Most-Researched Phytochemicals")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 — Species with the most associated chemicals
species_counts = df["plant_species"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
species_counts.plot.barh(ax=ax, color=sns.color_palette("mako", 20))
ax.set_xlabel("Number of Chemical Records")
ax.set_title("Top 20 Plant Species by Chemical Diversity")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Correlation heatmap of numeric columns
numeric_cols = [
    "pubmed_mentions_2026",
    # Enrichment columns (will be in v2.0 full dataset)
    # "clinical_trials_count_2026",
    # "chembl_bioactivity_count",
    # "patent_count_since_2020",
]

# For the sample, demonstrate with available numeric columns
available_numeric = [c for c in df.select_dtypes(include="number").columns]
print(f"Available numeric columns: {available_numeric}")

if len(available_numeric) >= 2:
    corr = df[available_numeric].corr()
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(corr, annot=True, cmap="coolwarm", center=0, ax=ax)
    ax.set_title("Feature Correlation Matrix")
    plt.tight_layout()
    plt.show()
else:
    print("Only one numeric column in sample — correlation matrix requires v2.0 full dataset.")
    print("Upgrade at https://ethno-api.com for 4 numeric enrichment columns.")

In [ ]:
# Cell 7 — Quick ML feature engineering example
# Demonstrates how to prepare the data for a classifier

# Binary target: does this compound have PubMed research?
df["has_research"] = (df["pubmed_mentions_2026"] > 0).astype(int)

# One-hot encode top species (as features)
top_species = df["plant_species"].value_counts().head(10).index
for sp in top_species:
    df[f"species_{sp.replace(' ', '_')}"] = (df["plant_species"] == sp).astype(int)

feature_cols = [c for c in df.columns if c.startswith("species_")]
X = df[feature_cols]
y = df["has_research"]

print(f"Features: {X.shape[1]} species indicators")
print(f"Target distribution:\n{y.value_counts().to_dict()}")
print("\nReady for sklearn, XGBoost, or any ML framework!")
print("\n📦 Full dataset: 104,388 records × 8 columns → https://ethno-api.com")